# Documents Summarizer

In [ ]:
from crewai import Crew, Agent, Task, LLM, Process
from crewai_tools import FileReadTool
import os
# Set up API key
os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"


### LLMs

In [3]:
summarizer_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.7,  # Balanced creativity for summarization
    max_tokens=500    # Limit the output length
)

manager_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,  # Factual and deterministic
    max_tokens=200    # Short feedback or approval responses
)

### Tools

In [4]:
file_read_tool = FileReadTool()

### Agents

In [5]:
# Summarizer Agent
summarizer = Agent(
    role="Text Summarizer",
    goal="Summarize the text in the file {path_to_file} into a concise and coherent summary.",
    backstory="A highly skilled summarizer specialized in condensing long texts.",
    tools=[file_read_tool],
    llm=summarizer_llm,
    verbose=True,
)

# Manager Agent
manager = Agent(
    role="Content Quality Manager",
    goal="Ensure that the summary provided is realistic and meets quality standards.",
    backstory="An experienced manager skilled in evaluating text summaries.",
    llm=manager_llm,
    verbose=True,
)

### Tasks

In [6]:
# Summarizer Task
summarize_task = Task(
    name="Summarize Text Task",
    agent=summarizer,
    description="Read the given text file and generate a concise summary.",
    expected_output="A brief and coherent summary of the text."
)

# Manager Task
validate_task = Task(
    name="Validate Summary Task",
    agent=manager,
    description=(
        "Evaluate the summary for realism and coherence. "
        "If it lacks realism or clarity, request a revision from the Summarizer."
    ),
    expected_output="Approval or specific feedback for improvement."
)

### Crew

In [7]:
crew = Crew(
    agents=[summarizer],
    tasks=[summarize_task],
    process=Process.hierarchical,
    manager_agent=manager
)

### Main

In [8]:
result = crew.kickoff(inputs={"path_to_file": "src/sample_text.txt"})
print("Final Output:")
print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Quality Manager                                                                                 │
│                                                                                                                 │
│  Task: Read the given text file and generate a concise summary.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_a_files_content executed with result: Error: File not found at path: path/to/your/textfile.txt...


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Text Summarizer                                                                                         │
│                                                                                                                 │
│  Task: Read the given text file and generate a concise summary.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_a_files_content executed with result: Error: File not found at path: src/sample_text.txt...
Tool read_a_files_content executed with result (from cache): Error: File not found at path: src/sample_text.txt...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Text Summarizer                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  File not found at the specified path.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Text Summarizer                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I currently cannot access the file at the specified path `src/sample_text.txt`. Please check if the file       │
│  exists at this location or provide an alternative file path for summarization.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool ask_question_to_coworker executed with result: I currently cannot access the file at the specified path `src/sample_text.txt`. Please check if the file exists at this location or provide an alternative file path for summarization....


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Quality Manager                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The requirements are not met as the file cannot be accessed at the specified path. There is no further action  │
│  to take without an alternative file path.                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final Output:
The requirements are not met as the file cannot be accessed at the specified path. There is no further action to take without an alternative file path.
